# AI Music OS — Kokoro TTS V5 (Colab Safe)

This notebook runs Kokoro TTS on Google Colab, including Python 3.13 runtimes.

Features:
- Google Drive persistent cache
- Kokoro from upstream GitHub on Python 3.13+
- Misaki phonemizer
- Required num2words dependency
- No forced NumPy/SciPy/Torch upgrades
- CPU/GPU detection
- WAV generation
- Gradio interface


In [ ]:
# STEP 1 — Google Drive + system dependency

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import subprocess

ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / 'kokoro'

CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE / 'transformers')
        
print('Python:', sys.version)
print('Python executable:', sys.executable)
print('Drive root:', ROOT)
print('Output folder:', OUT)

subprocess.run(
    ['apt-get', '-qq', 'update'],
    check=True
)

subprocess.run(
    ['apt-get', '-qq', '-y', 'install', 'espeak-ng'],
    check=True
)

print('espeak-ng: OK')


In [ ]:
# STEP 2 — Inspect existing Colab core packages
# IMPORTANT: We do NOT upgrade NumPy/SciPy/Torch here.

import sys
import numpy
import scipy
        
print('Python:', sys.version)
print('NumPy:', numpy.__version__)
print('SciPy:', scipy.__version__)

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
except Exception as e:
    print('PyTorch check error:', repr(e))


In [ ]:
# STEP 3 — Install Kokoro + Misaki safely
        
        import subprocess
        import sys
        
        def pip_install(*packages, no_deps=False):
        	cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir']
        	if no_deps:
        		cmd.append('--no-deps')
        	cmd.extend(packages)
        	print('Installing:', ' '.join(packages))
        	subprocess.run(cmd, check=True)
        
        if sys.version_info >= (3, 13):
        	print('Python 3.13+ detected.')
        	print('Installing upstream Kokoro + Misaki without changing Colab core stack.')
        
        	pip_install(
        		'git+https://github.com/hexgrad/misaki.git',
        		no_deps=True
        	)
        
        	pip_install(
        		'git+https://github.com/hexgrad/kokoro.git',
        		no_deps=True
        	)
        
        else:
        	print('Python <=3.12 detected.')
        	pip_install('kokoro>=0.9.4')
        
        # Runtime dependencies that are safe to install explicitly.
        pip_install(
        	'num2words',
        	'loguru',
        	'huggingface_hub',
        	'soundfile'
        )
        
        print('Kokoro + Misaki installation completed.')


In [ ]:
# STEP 4 — Verify required dependencies
        
        import numpy as np
        import scipy
        import torch
        
        print('NumPy:', np.__version__)
        print('SciPy:', scipy.__version__)
        print('PyTorch:', torch.__version__)
        print('CUDA available:', torch.cuda.is_available())
        
        if torch.cuda.is_available():
        	print('GPU:', torch.cuda.get_device_name(0))
        else:
        	print('CPU mode: GPU is not available.')
        
        # Check num2words explicitly because Misaki requires it.
        from num2words import num2words
        print('num2words: OK')
        print('num2words test:', num2words(123))
        
        # Check Transformers dependency.
        from transformers import AlbertModel
        print('Transformers AlbertModel: OK')


In [ ]:
# STEP 5 — Load Kokoro pipeline
        
        import kokoro
        from kokoro import KPipeline
        
        print('Kokoro module:', kokoro.__file__)
        
        pipeline = KPipeline(lang_code='a')
        
        print('Kokoro pipeline: OK')
        print('Kokoro is ready.')


In [ ]:
# STEP 6 — Kokoro audio generation
        
        import numpy as np
        import soundfile as sf
        import time
        
        VOICE_CHOICES = [
        	'af_heart',
        	'af_bella',
        	'af_nicole',
        	'af_sarah',
        	'af_sky',
        	'am_adam',
        	'am_michael',
        	'bf_emma',
        	'bf_isabella',
        	'bm_george',
        	'bm_lewis'
        ]
        
        SAMPLE_RATE = 24000
        
        def generate_audio(text, voice='af_heart'):
        	text = (text or '').strip()
        
        	if not text:
        		raise ValueError('Text is empty.')
        
        	if voice not in VOICE_CHOICES:
        		raise ValueError(f'Unsupported voice: {voice}')
        
        	chunks = []
        
        	print('Generating with Kokoro...')
        	print('Voice:', voice)
        
        	for item in pipeline(text, voice=voice):
        		if isinstance(item, tuple) and len(item) >= 3:
        			audio = item[2]
        		elif isinstance(item, dict):
        			audio = item.get('audio')
        		else:
        			audio = getattr(item, 'audio', None)
        
        		if audio is None:
        			raise RuntimeError(
        				f'Unsupported Kokoro output type: {type(item)}'
        			)
        
        		chunks.append(np.asarray(audio))
        
        	if not chunks:
        		raise RuntimeError('Kokoro returned no audio.')
        
        	audio = np.concatenate(chunks)
        
        	filename = f'kokoro_{int(time.time())}.wav'
        	path = OUT / filename
        
        	sf.write(path, audio, SAMPLE_RATE)
        
        	print('Audio saved:', path)
        	print('Duration:', round(len(audio) / SAMPLE_RATE, 2), 'seconds')
        
        	return str(path)
        
        print('Generation function: OK')


In [ ]:
# STEP 7 — Smoke test
        
        test_file = generate_audio(
        	'Hello. This is a test of the AI Music OS Kokoro voice generator.',
        	'af_heart'
        )
        
        print('=====================================')
        print('KOKORO TEST PASSED')
        print('File:', test_file)
        print('=====================================')


In [ ]:
# STEP 8 — Gradio UI
        
        import gradio as gr
        
        def ui_generate(text, voice):
        	try:
        		return generate_audio(text, voice)
        	except Exception as e:
        		raise gr.Error(str(e))
        
        demo = gr.Interface(
        	fn=ui_generate,
        	inputs=[
        		gr.Textbox(
        			lines=8,
        			label='Text',
        			value='Hello. This is the AI Music OS Kokoro voice generator.'
        		),
        		gr.Dropdown(
        			choices=VOICE_CHOICES,
        			value='af_heart',
        			label='Voice'
        		)
        	],
        	outputs=gr.Audio(
        		label='Generated audio',
        		type='filepath'
        	),
        	title='AI Music OS — Kokoro TTS V5',
        	description='Kokoro text-to-speech with Google Drive persistence.'
        )
        
        print('Launching Gradio...')
        demo.launch(share=True, debug=True)
